In [1]:
!pip3 install -q streamlit faiss-cpu sentence-transformers

In [2]:
!pip3 install -q streamlit_jupyter


In [3]:
import streamlit as st
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

# Load the embedding model
model = SentenceTransformer("distilbert-base-nli-stsb-mean-tokens", device="mps" if torch.backends.mps.is_available() else "cpu")

# Predefined Q&A dataset
data = [
    {"question": "What does the eligibility verification agent (EVA) do?", "answer": "EVA automates the process of verifying a patient’s eligibility and benefits information in real-time, eliminating manual data entry errors and reducing claim rejections."},
    {"question": "What does the claims processing agent (CAM) do?", "answer": "CAM streamlines the submission and management of claims, improving accuracy, reducing manual intervention, and accelerating reimbursements."},
    {"question": "How does the payment posting agent (PHIL) work?", "answer": "PHIL automates the posting of payments to patient accounts, ensuring fast, accurate reconciliation of payments and reducing administrative burden."},
    {"question": "Tell me about Thoughtful AI's Agents.", "answer": "Thoughtful AI provides a suite of AI-powered automation agents designed to streamline healthcare processes. These include Eligibility Verification (EVA), Claims Processing (CAM), and Payment Posting (PHIL), among others."},
    {"question": "What are the benefits of using Thoughtful AI's agents?", "answer": "Using Thoughtful AI's Agents can significantly reduce administrative costs, improve operational efficiency, and reduce errors in critical processes like claims management and payment posting."},
    {"question": "What does the eligibility verification agent (EVA) do?", "answer": "EVA automates the process of verifying a patient’s eligibility and benefits information in real-time, eliminating manual data entry errors and reducing claim rejections."},
    {"question": "What does the claims processing agent (CAM) do?", "answer": "CAM streamlines the submission and management of claims, improving accuracy, reducing manual intervention, and accelerating reimbursements."},
    {"question": "How does the payment posting agent (PHIL) work?", "answer": "PHIL automates the posting of payments to patient accounts, ensuring fast, accurate reconciliation of payments and reducing administrative burden."},
    {"question": "Tell me about Thoughtful AI's Agents.", "answer": "Thoughtful AI provides a suite of AI-powered automation agents designed to streamline healthcare processes. These include Eligibility Verification (EVA), Claims Processing (CAM), and Payment Posting (PHIL), among others."},
    {"question": "What are the benefits of using Thoughtful AI's agents?", "answer": "Using Thoughtful AI's Agents can significantly reduce administrative costs, improve operational efficiency, and reduce errors in critical processes like claims management and payment posting."},
    {"question": "What does the eligibility verification agent (EVA) do?", "answer": "EVA automates the process of verifying a patient’s eligibility and benefits information in real-time, eliminating manual data entry errors and reducing claim rejections."},
    {"question": "What does the claims processing agent (CAM) do?", "answer": "CAM streamlines the submission and management of claims, improving accuracy, reducing manual intervention, and accelerating reimbursements."},
    {"question": "How does the payment posting agent (PHIL) work?", "answer": "PHIL automates the posting of payments to patient accounts, ensuring fast, accurate reconciliation of payments and reducing administrative burden."},
    {"question": "Tell me about Thoughtful AI's Agents.", "answer": "Thoughtful AI provides a suite of AI-powered automation agents designed to streamline healthcare processes. These include Eligibility Verification (EVA), Claims Processing (CAM), and Payment Posting (PHIL), among others."},
    {"question": "What are the benefits of using Thoughtful AI's agents?", "answer": "Using Thoughtful AI's Agents can significantly reduce administrative costs, improve operational efficiency, and reduce errors in critical processes like claims management and payment posting."}
]


In [4]:
# Encode questions and create FAISS index
questions = [item["question"] for item in data]
question_embeddings = np.array(model.encode(questions)).astype('float32')

# Create the FAISS index
dimension = question_embeddings.shape[1]
nlist = 10  # Number of clusters
quantizer = faiss.IndexFlatL2(dimension)
index = faiss.IndexIVFFlat(quantizer, dimension, nlist, faiss.METRIC_L2)

# Train the index
index.train(question_embeddings)
index.add(question_embeddings)

WARNING clustering 15 points to 10 centroids: please provide at least 390 training points


In [10]:

def find_best_answer(query):
    """
    This function performs semantic search using an Approximate Nearest Neighbor (ANN) algorithm
    powered by FAISS (Facebook AI Similarity Search). It encodes the user's query into a vector
    using a pre-trained SentenceTransformer model, then searches for the most similar question
    embedding in the FAISS index. The closest match is determined based on L2 distance in the
    embedding space, allowing for efficient and scalable retrieval even with large datasets.
    
    FAISS (Facebook AI Similarity Search) implements efficient similarity search using vector quantization and clustering techniques.
    In this implementation, we use an IndexIVFFlat index, which works in two main steps:
    1. Clustering (Inverted File Index): During training, FAISS partitions the embedding space into 'nlist' clusters (here, 10) using k-means.
       Each cluster is represented by a centroid, and each question embedding is assigned to its nearest centroid.
    2. Search (Efficient Tree-like Retrieval): When a query comes in, FAISS first finds the closest centroid(s) to the query embedding.
       Instead of searching all vectors, it only searches within the vectors assigned to those centroids (inverted lists), drastically reducing the search space.
       This is analogous to a tree search, where the search is narrowed down to relevant branches (clusters) before checking individual leaves (embeddings).
       The L2 distance metric is used to measure similarity between the query and stored embeddings.
    This two-level approach (coarse search for clusters, then fine search within clusters) enables fast and scalable retrieval, even for large datasets.

    Args:
        query (str): The user's input question to search for the best matching answer.

    Returns:
        str: The answer corresponding to the most similar question in the dataset. If no suitable
        match is found, returns a default message indicating no information is available.
    """
    query_embedding = np.array(model.encode([query])).astype('float32')
    D, I = index.search(query_embedding, 1)  # Find the closest match
    
    if D[0][0]:  # Adjust threshold as needed
        return data[I[0][0]]["answer"]
    else:
        return "I'm sorry, but I don't have information on that. Could you rephrase or ask something else?"



In [16]:
find_best_answer("What's Thoughtful AI's Agents.")

'Thoughtful AI provides a suite of AI-powered automation agents designed to streamline healthcare processes. These include Eligibility Verification (EVA), Claims Processing (CAM), and Payment Posting (PHIL), among others.'

In [17]:
!pip install -q streamlit_jupyter streamlit

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [6]:

# To run Streamlit from a Jupyter notebook, use the `streamlit_jupyter` package:
# 1. Install with: pip install streamlit_jupyter
# 2. Then, use the following pattern in a notebook cell:
# 
import streamlit as st

from streamlit_jupyter import StreamlitPatcher, tqdm

sp = StreamlitPatcher()
sp.jupyter()

# my_app():

st.title("Thoughtful AI Support Agent")
st.write("Ask me anything about Thoughtful AI!")
user_query = st.text_input("Your question:")
if user_query:
    response = find_best_answer(user_query)
    st.write("### Response:")
    st.write(response)
# your Streamlit code here

# 
# See cell 9 below for an example.


AttributeError: module 'streamlit' has no attribute 'experimental_data_editor'

In [ ]:

from streamlit_jupyter import st_jupyter

# Streamlit UI Demo
def chatbot_demo():
    st.title("Thoughtful AI Support Agent")
    st.write("Ask me anything about Thoughtful AI!")
    user_query = st.text_input("Your question:")
    if user_query:
        response = find_best_answer(user_query)
        st.write("### Response:")
        st.write(response)


st_jupyter(chatbot_demo)


ImportError: cannot import name 'st_jupyter' from 'streamlit_jupyter' (/opt/homebrew/anaconda3/envs/python-notebook/lib/python3.9/site-packages/streamlit_jupyter/__init__.py)